In [1]:
import pandas as pd
import numpy as np

patients = pd.read_csv("patients.csv")
encounters = pd.read_csv("encounters.csv")
procedures = pd.read_csv("procedures.csv")
payers = pd.read_csv("payers.csv")
organizations = pd.read_csv("organizations.csv")
data_dictionary = pd.read_csv("data_dictionary.csv")

# Convert Date Columns

In [2]:
patients["BIRTHDATE"] = pd.to_datetime(patients["BIRTHDATE"], errors="coerce")
patients["DEATHDATE"] = pd.to_datetime(patients["DEATHDATE"], errors="coerce")

encounters["START"] = pd.to_datetime(encounters["START"], errors="coerce")
encounters["STOP"] = pd.to_datetime(encounters["STOP"], errors="coerce")

procedures["START"] = pd.to_datetime(procedures["START"], errors="coerce")
procedures["STOP"] = pd.to_datetime(procedures["STOP"], errors="coerce")

# Clean ZIP Codes

In [3]:
patients["ZIP"] = patients["ZIP"].astype("Int64").astype(str)
payers["ZIP"] = payers["ZIP"].astype("Int64").astype(str)
organizations["ZIP"] = organizations["ZIP"].astype(str)

In [4]:
patients["ZIP"] = patients["ZIP"].replace("<NA>", "Unknown")
payers["ZIP"] = payers["ZIP"].replace("<NA>", "Unknown")

# Handle Missing Values

In [5]:
patients["MARITAL"] = patients["MARITAL"].fillna("Unknown")
patients["SUFFIX"] = patients["SUFFIX"].fillna("Not Available")
patients["MAIDEN"] = patients["MAIDEN"].fillna("Not Available")
patients["ZIP"] = patients["ZIP"].fillna("Unknown")

encounters["REASONCODE"] = encounters["REASONCODE"].fillna("Not Available")
encounters["REASONDESCRIPTION"] = encounters["REASONDESCRIPTION"].fillna("Not Available")

procedures["REASONCODE"] = procedures["REASONCODE"].fillna("Not Available")
procedures["REASONDESCRIPTION"] = procedures["REASONDESCRIPTION"].fillna("Not Available")

payers["ADDRESS"] = payers["ADDRESS"].fillna("Not Available")
payers["CITY"] = payers["CITY"].fillna("Not Available")
payers["STATE_HEADQUARTERED"] = payers["STATE_HEADQUARTERED"].fillna("Not Available")
payers["ZIP"] = payers["ZIP"].fillna("Unknown")
payers["PHONE"] = payers["PHONE"].fillna("Not Available")

In [7]:
# Creating Length of stay column
encounters["length_of_stay_hours"] = (
    encounters["STOP"] - encounters["START"]
).dt.total_seconds() / 3600

# Length of stay in days
encounters["length_of_stay_days"] = encounters["length_of_stay_hours"] / 24

In [8]:
encounters[["START", "STOP", "length_of_stay_hours", "length_of_stay_days"]].head()

,START,STOP,length_of_stay_hours,length_of_stay_days
0,2011-01-02 09:26:36+00:00,2011-01-02 12:58:36+00:00,3.533333,0.147222
1,2011-01-03 05:44:39+00:00,2011-01-03 06:01:42+00:00,0.284167,0.011840
2,2011-01-03 14:32:11+00:00,2011-01-03 14:47:11+00:00,0.250000,0.010417
3,2011-01-03 16:24:45+00:00,2011-01-03 16:39:45+00:00,0.250000,0.010417
4,2011-01-03 17:36:53+00:00,2011-01-03 17:51:53+00:00,0.250000,0.010417


# Create Year, Month, and Year-Month Columns

In [10]:
encounters["encounter_year"] = encounters["START"].dt.year
encounters["encounter_month"] = encounters["START"].dt.month
encounters["encounter_month_name"] = encounters["START"].dt.month_name()

encounters["encounter_year_month"] = encounters["START"].dt.strftime("%Y-%m")

# Create Patient Age

In [11]:
# First merge encounters with patient birthdate:
encounters_patient = encounters.merge(
    patients[["Id", "BIRTHDATE", "GENDER", "RACE", "ETHNICITY", "MARITAL"]],
    left_on="PATIENT",
    right_on="Id",
    how="left",
    suffixes=("", "_patient")
)

In [12]:
#creating Age column
encounters_patient["age_at_encounter"] = (
    encounters_patient["START"].dt.year - encounters_patient["BIRTHDATE"].dt.year
)

In [13]:
#Creating Age Group
def age_group(age):
    if pd.isna(age):
        return "Unknown"
    elif age <= 18:
        return "0-18"
    elif age <= 35:
        return "19-35"
    elif age <= 50:
        return "36-50"
    elif age <= 65:
        return "51-65"
    else:
        return "65+"

encounters_patient["age_group"] = encounters_patient["age_at_encounter"].apply(age_group)

In [14]:
encounters_patient[["PATIENT", "START", "BIRTHDATE", "age_at_encounter", "age_group"]].head()

,PATIENT,START,BIRTHDATE,age_at_encounter,age_group
0,3de74169-7f67-9304-91d4-757e0f3a14d2,2011-01-02 09:26:36+00:00,1928-08-25,83,65+
1,d9ec2e44-32e9-9148-179a-1653348cc4e2,2011-01-03 05:44:39+00:00,1964-01-05,47,36-50
2,73babadf-5b2b-fee7-189e-6f41ff213e01,2011-01-03 14:32:11+00:00,1924-06-30,87,65+
3,3b46a0b7-0f34-9b9a-c319-ace4a1f58c0b,2011-01-03 16:24:45+00:00,1923-05-21,88,65+
4,fa006887-d93c-d302-8b89-f3c25f88c0e1,2011-01-03 17:36:53+00:00,1952-11-02,59,51-65


# Create Insurance Coverage Flag

In [15]:
# For encounters
encounters["is_covered_by_insurance"] = np.where(
    encounters["PAYER_COVERAGE"] > 0,
    "Covered",
    "Not Covered"
)

In [16]:
# For procedures, we need to connect procedures with encounter payer coverage:

procedures_with_coverage = procedures.merge(
    encounters[["Id", "PAYER", "PAYER_COVERAGE", "TOTAL_CLAIM_COST"]],
    left_on="ENCOUNTER",
    right_on="Id",
    how="left",
    suffixes=("", "_encounter")
)

procedures_with_coverage["is_covered_by_insurance"] = np.where(
    procedures_with_coverage["PAYER_COVERAGE"] > 0,
    "Covered",
    "Not Covered"
)

# Readmission in Hospital

In [17]:
# A patient having another encounter after a previous encounter.

encounters_sorted = encounters.sort_values(["PATIENT", "START"])

encounters_sorted["previous_encounter_date"] = encounters_sorted.groupby("PATIENT")["START"].shift(1)

encounters_sorted["days_since_previous_visit"] = (
    encounters_sorted["START"] - encounters_sorted["previous_encounter_date"]
).dt.days

encounters_sorted["is_readmission"] = np.where(
    encounters_sorted["previous_encounter_date"].notna(),
    "Readmission",
    "First Admission"
)

In [18]:
# Readmission in between 30 days
encounters_sorted["is_30_day_readmission"] = np.where(
    encounters_sorted["days_since_previous_visit"].between(1, 30),
    "Yes",
    "No"
)

In [19]:
patients.info()
encounters.info()
procedures.info()

encounters[["length_of_stay_hours", "length_of_stay_days"]].describe()

encounters["is_covered_by_insurance"].value_counts()

encounters_sorted["is_readmission"].value_counts()

encounters_sorted["is_30_day_readmission"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 20 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Id          974 non-null    object        
 1   BIRTHDATE   974 non-null    datetime64[ns]
 2   DEATHDATE   154 non-null    datetime64[ns]
 3   PREFIX      974 non-null    object        
 4   FIRST       974 non-null    object        
 5   LAST        974 non-null    object        
 6   SUFFIX      974 non-null    object        
 7   MAIDEN      974 non-null    object        
 8   MARITAL     974 non-null    object        
 9   RACE        974 non-null    object        
 10  ETHNICITY   974 non-null    object        
 11  GENDER      974 non-null    object        
 12  BIRTHPLACE  974 non-null    object        
 13  ADDRESS     974 non-null    object        
 14  CITY        974 non-null    object        
 15  STATE       974 non-null    object        
 16  COUNTY      974 non-null  

is_30_day_readmission
Yes    16234
No     11657
Name: count, dtype: int64

In [20]:
encounters_sorted = encounters.sort_values(["PATIENT", "START"]).copy()

encounters_sorted["previous_encounter_date"] = encounters_sorted.groupby("PATIENT")["START"].shift(1)

encounters_sorted["days_since_previous_visit"] = (
    encounters_sorted["START"] - encounters_sorted["previous_encounter_date"]
).dt.days

encounters_sorted["is_readmission"] = np.where(
    encounters_sorted["previous_encounter_date"].notna(),
    "Readmission",
    "First Admission"
)

encounters_sorted["is_30_day_readmission"] = np.where(
    encounters_sorted["days_since_previous_visit"].between(1, 30),
    "Yes",
    "No"
)

In [21]:
encounters_sorted[[
    "PATIENT",
    "START",
    "previous_encounter_date",
    "days_since_previous_visit",
    "is_readmission",
    "is_30_day_readmission"
]].head(10)

,PATIENT,START,previous_encounter_date,days_since_previous_visit,is_readmission,is_30_day_readmission
1292,002bc307-2fff-04ba-161b-98cce123e226,2011-12-22 19:30:33+00:00,NaT,NaN,First Admission,No
1403,002bc307-2fff-04ba-161b-98cce123e226,2012-01-19 19:30:33+00:00,2011-12-22 19:30:33+00:00,28.0,Readmission,Yes
8398,002bc307-2fff-04ba-161b-98cce123e226,2014-06-19 19:30:33+00:00,2012-01-19 19:30:33+00:00,882.0,Readmission,No
9343,002bc307-2fff-04ba-161b-98cce123e226,2014-10-30 19:30:33+00:00,2014-06-19 19:30:33+00:00,133.0,Readmission,No
9579,002bc307-2fff-04ba-161b-98cce123e226,2014-11-29 19:30:33+00:00,2014-10-30 19:30:33+00:00,30.0,Readmission,Yes
9808,002bc307-2fff-04ba-161b-98cce123e226,2014-12-29 19:30:33+00:00,2014-11-29 19:30:33+00:00,30.0,Readmission,Yes
10007,002bc307-2fff-04ba-161b-98cce123e226,2015-01-28 19:30:33+00:00,2014-12-29 19:30:33+00:00,30.0,Readmission,Yes
10222,002bc307-2fff-04ba-161b-98cce123e226,2015-02-27 19:30:33+00:00,2015-01-28 19:30:33+00:00,30.0,Readmission,Yes
10437,002bc307-2fff-04ba-161b-98cce123e226,2015-03-29 19:30:33+00:00,2015-02-27 19:30:33+00:00,30.0,Readmission,Yes
10629,002bc307-2fff-04ba-161b-98cce123e226,2015-04-28 19:30:33+00:00,2015-03-29 19:30:33+00:00,30.0,Readmission,Yes


In [22]:
patients.to_csv("cleaned_patients.csv", index=False)
encounters.to_csv("cleaned_encounters.csv", index=False)
procedures.to_csv("cleaned_procedures.csv", index=False)
payers.to_csv("cleaned_payers.csv", index=False)
organizations.to_csv("cleaned_organizations.csv", index=False)
encounters_sorted.to_csv("cleaned_encounters_with_readmission.csv", index=False)
procedures_with_coverage.to_csv("cleaned_procedures_with_coverage.csv", index=False)